# Serpientes y Escaleras y Ratón - Regla de rebote

## Especificaciones

### Problema 1: Serpientes y escaleras (casillas 1..20, meta=20)
- Escaleras: 3 → 11 , 15 → 19
- Serpientes: 17 → 10 , 13 → 4
- Rebote: si el dado supera la casilla 20 entonces rebota asi la nueva posición  = 40 − (casilla actual + dado)
- Calcularemos el numero esperado de tiradas desde la casilla 1.

### Problema 2: El raton y la comida (casillas 0..8)
- Comida (victoria) en casilla 7
- Shock (derrota) en casilla 8
- Rebote: si el dado supera la casilla 8 entonces la nueva posición = 16 − (casilla actual + dado)
- Se calcula la probabilidad de alcanzar la comida (7) antes que el shock (8) desde la casilla 0.

## Ecuaciones del modelo (LaTeX)

Para los estados transitorios $S$ (1..19 en el juego 1 o bien 0..6 en el juego 2):

**Valor esperado de tiradas:**
$$
E_i = 1 + \sum_{j \in S} P_{ij} E_j \quad\Rightarrow\quad (\mathbf{I} - \mathbf{Q})\mathbf{E} = \mathbf{1}
$$

**Probabilidad de éxito (comida antes que shock):**
$$
p_i = \sum_{j \in S} P_{ij} p_j + \sum_{k \in \text{win}} P_{ik}
\quad\Rightarrow\quad (\mathbf{I} - \mathbf{Q})\mathbf{p} = \mathbf{b}
$$
donde $b_i = P_{i,\text{comida}}$.

In [2]:
import sympy as sp
import random

def p1():
    sl = {3:11, 15:19, 17:10, 13:4}   
    n = 20
    Q = [[0]*19 for _ in range(19)]
    for i in range(1,20):         
        for d in range(1,7):
            nxt = i + d
            if nxt > n:
                nxt = 2*n - nxt    
            else:
                nxt = sl.get(nxt, nxt)
            if nxt == n:           
                continue
            if 1 <= nxt <= 19:
                Q[i-1][nxt-1] += 1/6
    Qsp = sp.Matrix(Q)
    I = sp.eye(19)
    E = (I - Qsp).inv() * sp.Matrix([1]*19)
    return sp.simplify(E[0])

def sim1(N=100000):
    sl = {3:11, 15:19, 17:10, 13:4}
    total = 0
    for _ in range(N):
        pos = 1
        tiros = 0
        while pos < 20:
            d = random.randint(1,6)
            tiros += 1
            nxt = pos + d
            if nxt > 20:
                nxt = 40 - nxt      # rebote
            nxt = sl.get(nxt, nxt)
            pos = nxt
        total += tiros
    return total / N

def p2():
    """Analítico: probabilidad de llegar a comida (7) antes que shock (8) desde 0"""
    win = 7
    lose = 8
    ntrans = 7   
    Q = [[0]*ntrans for _ in range(ntrans)]
    b = [0]*ntrans
    for i in range(ntrans):
        for d in range(1,7):
            nxt = i + d
            if nxt > lose:
                nxt = 2*lose - nxt   
            if nxt == win:
                b[i] += 1/6
                continue
            if nxt == lose:
                continue   
            Q[i][nxt] += 1/6
    Qsp = sp.Matrix(Q)
    I = sp.eye(ntrans)
    bsp = sp.Matrix(b)
    p = (I - Qsp).inv() * bsp
    return sp.simplify(p[0])   

def sim2(N=100000):
    win, lose = 7, 8
    exitos = 0
    for _ in range(N):
        pos = 0
        while True:
            d = random.randint(1,6)
            nxt = pos + d
            if nxt > lose:
                nxt = 2*lose - nxt
            if nxt == win:
                exitos += 1
                break
            if nxt == lose:
                break
            pos = nxt
    return exitos / N

def menu():
    while True:
        print("\n" + "="*50)
        print(" MENU (REGLA DE REBOTE - CORREGIDO)")
        print(" 1 - Serpientes y escaleras (E[tiradas] desde 1)")
        print(" 2 - Ratón y comida (probabilidad desde 0 hacia comida 7)")
        print(" 0 - Salir")
        op = input("Opción: ").strip()
        try:
            op = int(op)
        except:
            print("Entrada inválida. Intente de nuevo.")
            continue
        if op == 0:
            print("Adiocito")
            break
        elif op == 1:
            print("Resolución analitica")
            try:
                exp = p1()
                print(f"E[tiradas] exacto = {exp}")
                print(f"≈ {float(exp):.6f}")
            except Exception as e:
                print(f"Error analítico: {e}")
            print("Simulacion")
            try:
                s = sim1(200000)
                print(f"Promedio simulado = {s:.6f}")
            except Exception as e:
                print(f"Error simulación: {e}")
        elif op == 2:
            print("Resolución analitica")
            try:
                prob = p2()
                print(f"Probabilidad exacta = {prob}")
                print(f"≈ {float(prob):.6f}")
            except Exception as e:
                print(f"Error analítico: {e}")
            print("Simulacion")
            try:
                s = sim2(200000)
                print(f"Probabilidad simulada = {s:.6f}")
            except Exception as e:
                print(f"Error simulación: {e}")
        else:
            print("Opción no válida. Elija 1, 2 o 0.")

if __name__ == "__main__":
    menu()


 MENU (REGLA DE REBOTE - CORREGIDO)
 1 - Serpientes y escaleras (E[tiradas] desde 1)
 2 - Ratón y comida (probabilidad desde 0 hacia comida 7)
 0 - Salir


Opción:  1


Resolución analitica
E[tiradas] exacto = 11.8126651901296
≈ 11.812665
Simulacion
Promedio simulado = 13.560060

 MENU (REGLA DE REBOTE - CORREGIDO)
 1 - Serpientes y escaleras (E[tiradas] desde 1)
 2 - Ratón y comida (probabilidad desde 0 hacia comida 7)
 0 - Salir


Opción:  1


Resolución analitica
E[tiradas] exacto = 11.8126651901296
≈ 11.812665
Simulacion
Promedio simulado = 13.568285

 MENU (REGLA DE REBOTE - CORREGIDO)
 1 - Serpientes y escaleras (E[tiradas] desde 1)
 2 - Ratón y comida (probabilidad desde 0 hacia comida 7)
 0 - Salir


Opción:  2


Resolución analitica
Probabilidad exacta = 0.665123456790123
≈ 0.665123
Simulacion
Probabilidad simulada = 0.662205

 MENU (REGLA DE REBOTE - CORREGIDO)
 1 - Serpientes y escaleras (E[tiradas] desde 1)
 2 - Ratón y comida (probabilidad desde 0 hacia comida 7)
 0 - Salir


Opción:  0


Adiocito
